# **Infant Cry Sound Detection**

## **Abstract**
Please find the repository for this project: https://github.com/JayC-SF/COMP-432-Project
Please also note the 
Abstract here. Give an executive summary of your project: goal, methods, results, conclusions. Usually no more than 200 words.


## **Introduction**
This project attempts to solve the infant cry sound detection problem. In this modern day, many  

## **2. Methodology**
### **2.1 Preprocessing**
The audio files were preprocessed in to mel spectograms where they needed to be formatted to the same audio lengths. Most of audio lengths were around 10 seconds, however, there were a few samples that were around 6-8 seconds, with a few outliers with only 1-2 seconds. Since the maximum length of audio was 10 seconds, all samples with less durations were padded with empty sound to fit the durations. 

<div align="center">
    <img src="images/train_dataset_distribution.png" width=400>
    <img src="images/validation_dataset_distribution.png" width=400>
    <img src="images/test_dataset_duration_distribution.png" width=400>
</div>

The class imbalance was verified for the train, val and test splits, where for pretty much all splits there were roughly 50/50 InfantCry/Snoring balancing of the classes.

<div align="center">
    <img src="images/training_class_imbalance.png" width=400>
    <img src="images/validation_class_imbalance.png" width=400>
    <img src="images/testing_class_imbalance.png" width=400>
</div>


### **2.2 Models**
This project performs an analysis of 3 deep learning models: Classical CNN baseline, a Convolutional LSTM, and a Resnet model. 


#### **2.2.1 Classical CNN Architecture**
The ClassicCNN is a traditional sequential architecture designed for hierarchical feature extraction from audio spectrograms. 

It consists of a three-layer convolutional backbone where the depth progressively increases (32, 64, and 128 filters), while integrated Batch Normalization for stability and Max-Pooling layers to reduce spatial dimensions. 

The extracted features are flattened into a high-dimensional vector and processed through a 256-unit fully connected layer, which utilizes 30% Dropout during training to provide regularization before final classification.

<div align="center">
    <img src="images/classical_cnn_architecture.png">
</div>

#### **2.2.2 Convolutional LSTM (Best Model)**

The Convolutional LSTM follows the same convolution as the Classical CNN, however instead of applying two fully connected layers it goes through a a two layer LSTM with  `hidden_size=256` to parse the temporal aspect of the input. It can be seen later in this report how this piece does increase the performance of the model. A dropout of 0.3 was also added in this model to add regularization.

<div align="center">
    <img src="images/CLSTM_architecture.png">
</div>

#### **2.2.3 ResNet**

The ResNet architecture is by far the largest model in this report. This model consists of 4 layers of Residual Blocks. Please find the architecture of the residual block as below:

<div align="center">
    <img src="images/residualblockpng.png" width=400>
</div>

The residual block contains 2 paths, from which 1 contains 2 convolutional layers followed by batch normalization and relu activation functions and the other one contains only 1 convolutional layer and a single batch norm. The output of those two paths are then merged by adding the results of both paths and a final ReLU activation function is applied.

The architecture of the resnet model is depicted as below, demonstrating 4 layers of residual blocks.

<div align="center">
    <img src="images/resnet_architecture.png" width=1000>
</div>

#### **2.2.4 ML Pipeline**
Every model above have been trained with a baseline set of hyper parameters. They were then evaluated with their respective metrics like losses, accuracies (val & train) over epochs. The best epoch for the models. From those evaluations, a range of hyper parameters on learning rates and weight decay were determined to perform the hyper parameter tuning. 

The library `optuna` was leveraged for the hyper parameter tuning. This helped performing some pruning for some searches that were not relevant and helped speed up the fine tuning of the models.

## **3. Experimental Setup**
### **3.1 ICSD Dataset**
In this experiment, the ICSD dataset has been used to train all the models. The ICSD dataset consists of ~14, 000 audio samples with mainly 10 seconds duration. A few samples are not of the same durations where some are around 1-2 seconds with a few more that are 6-9 seconds. 

The total dataset consists of 3 main components: real strong, synthetic strong, and weak labeling.

**Real strong labeling**: Consists of real data that are strongly labeled where the exact time of the infant cry is recorded in the metadata.

**Syntehtic strong labeling**: Consists of synthetic (real infant cries merged with Urban sounds) data that are strongly labeled where the exact time of the infant cry is recorded in the metadata.

**Weak**: Consists of real data where the exact time of the infant cry is not recorded but its classification is noted as either infant cry or snoring.

Since the goal of this project is not to specifically determine the exact time of the infant cry These three components were taken into a single dataset In order to train the models. The ICSD dataset came with a predetermined train validation and test split performed by the owners of the dataset. Since this split was performed by current researchers it is assumed that the dataset does not contain any leaks between each split.

The dataset was preprocessed using the `librosa` library. Every audio samples were converted into mel spectograms. The mel spectogram convertion used a sample rate of 16000, 128 mels, and 512 hop length. 

| Parameter | Value | Explanation |
| :--- | :--- | :--- |
| **Sample Rate (`sr`)** | 16000 |The number of digital snapshots taken per second. It determines the maximum frequency captured (Nyquist frequency). |
| **Mel Bands (`n_mels`)** | 128 | The number of frequency bins on the vertical axis, scaled to match human auditory perception of pitch. |
| **Hop Length (`hop_length`)** | 512 | The number of audio samples the analysis window "steps" forward for each frame; defines the temporal density. Equivalent to taking snapshots of the audio every 23 milliseconds|

After being converted into mel spectograms, the samples were then normalized to a range of [0,1] using a [Min-Max Scaling Normalization](https://en.wikipedia.org/wiki/Feature_scaling#Rescaling_(min-max_normalization)). 


$$x'=\frac{x-min(x)}{max(x)-min(x)}$$

There were three main reasons the dataset was normalized:
1. Potentially better convergence to avoid exploding or vanishing gradients. 
2. Avoids providing links of loud sounds to crying sounds, but instead decode the texture and curves of an infant cry.
3. Combats sensitivity of ReLu activation function since the positive side of the activation function is a linear function.


After the first conversion to mel spectograms, the data is gathered together by either truncating or padding the audio files so they have the same time dimension length (313).

From the records, the data was never really truncated but only padded with empty sound (0-value) for the short duration ones.
The dataset is then converted into a single .npz file to contain all of the mel spectrograms into a single file. The conversion into a single file was to facilitate loading the dataset for training models.

### **3.2 Model Configurations**
An [Orchestrator](https://github.com/JayC-SF/COMP-432-Project/blob/main/src/train/orchestrator.py) class was created for reusability of the train, validation, and test pipelines onto the models.

All models were trained with early stopping and a patience of 15 to provide them a chance to get out of local minimums. They all shared Adam W optimizer as it was determined to be probably the better option as the optimizer for these models. The scheduler was used as well to determine the learning rates of the training. Not all models shared the same notably the Resnet model use the cosine annealing scheduler for the training.  The Cross Entropy Loss function is used even though the binary classification models are trained since the training pipeline can be extended for multi class classification eventually for further projects.

The training pipeline saves the history of the orchestrator in order to continue the training in case the training is halted. The history saved includes the training and validation losses, accuracies, optimizer, scheduler, best model weights, the current model, early stopping count and epoch.

Note as well as mentioned earlier The models were also fine tuned based on learning rate and weight decay only. The Optuna Library was used to perform the fine tuning of these hyperparameters. Each models are trained with 15 trials where some trials were pruned as some training instances were found to have better potential. This was particularly useful since a great search would have been too expensive to train in order to the hyperparameters. Additionally the Optina library took care of selecting the hyperparameters over a continuous interval.


#### **3.2.1 Classic CNN Hyperparameters**

The Classic CNN baseline model hyperparameters can be found in the table below. The learning rate chosen for this baseline was simply a classic baseline LR. We also add a light touch of weight decay to prevent overfitting of the model as well as a 0.3 or dropout rate to help with regularization. Note the scheduler patience of 5 to help converging the learning rate on validation loss plateaus.

| Hyperparameter | Value |
| :--- | :--- |
| **Optimizer** | AdamW |
| **Scheduler** | ReduceLROnPlateau |
| **Scheduler Patience** | 5 |
| **Scheduler Factor** | 0.1 |
| **Learning Rate** | 1e-3 |
| **Batch Size** | 64 |
| **Patience** | 15 |
| **Loss Function** | CrossEntropyLoss |
| **Dropout Rate** | 0.3 |
| **Weight Decay** | 5e-5 |
| **Input Shape** | (12284, 128, 313) |


| Hyperparameter | Low Bound | High Bound |
| :--- | :--- | :--- |
| ** Learning Rate** | 1e-5 | 1e-2 |
| ** Weight Decay** | 1e-6 | 1e-1 |

#### **3.2.2 Convolutional LSTM Hyperparameters**

| Parameter | Value |
| :--- | :--- |
| **Optimizer** | AdamW |
| **Scheduler** | ReduceLROnPlateau |
| **Scheduler Patience** | 5 |
| **Scheduler Factor** | 0.1 |
| **Learning Rate** | 5e-4 |
| **Batch Size** | 64 |
| **Patience** | 15 |
| **Loss Function** | CrossEntropyLoss |
| **Dropout Rate** | 0.3 |
| **Weight Decay** | 5e-4 |
| **Input Shape** | (12284, 128, 313) |



## **Experimental Results**
Describe here the main experimental results. Critically discuss them. Compare them with results available in the literature (if applicable).

In this section, you can add **text** and **figures**, **tables**, **plots**, and code. Make sure the code is runnable and replicable.

## **Conclusions**

Summarize what you could and could not conclude based on your experiments.
In this section, you can add **text**.



## **References**
You can add here the citations of books, websites, or academic papers, etc.